In [6]:
#cell 1: kiểm tra MediaPipe, OpenCV và CUDA cùng hoạt động trong train-gpu
import torch, cv2, mediapipe as mp
print("MediaPipe:", mp.__version__, "| OpenCV:", cv2.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0))

MediaPipe: 1.1.0 | OpenCV: 5.0.0
CUDA: True NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [7]:
#cell 2: quét Dataset_hand và dataset_labeled, đếm video theo lớp, xem fps / số khung / độ dài
import cv2
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")
exts = {".mp4", ".mov", ".avi", ".mkv"}

rows = []
for src in ["Dataset_hand", "dataset_labeled"]:
    for p in (BASE / src).rglob("*"):
        if p.suffix.lower() in exts:
            cap = cv2.VideoCapture(str(p))
            fps = cap.get(cv2.CAP_PROP_FPS)
            n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            rows.append(dict(nguon=src, lop=p.parent.name, file=p.name, fps=round(fps),
                             khung=n, giay=round(n / fps, 2) if fps else 0,
                             MB=round(p.stat().st_size / 1e6, 2)))

df = pd.DataFrame(rows)
print(df.groupby(["nguon", "lop"]).agg(so_video=("file", "count"), fps=("fps", "median"),
                                       giay_tb=("giay", "mean"), MB=("MB", "sum")))
df.head()

                          so_video    fps   giay_tb     MB
nguon           lop                                       
Dataset_hand    Paper           27   60.0  1.058148  44.10
                Rock            27   60.0  1.060000  43.52
                Scissors        27   60.0  1.069259  45.06
dataset_labeled paper           48  120.0  0.933750  73.08
                rock            43  120.0  1.019302  64.91
                scissors        43  120.0  1.187674  70.10


,nguon,lop,file,fps,khung,giay,MB
0,Dataset_hand,Paper,VID_20261002074120297.mp4,60,68,1.13,1.60
1,Dataset_hand,Paper,VID_20261002075216637.mp4,60,63,1.05,1.54
2,Dataset_hand,Paper,VID_20261002075637370.mp4,60,61,1.02,1.37
3,Dataset_hand,Paper,VID_20261002080154187.mp4,60,63,1.05,1.54
4,Dataset_hand,Paper,VID_20261002080442985.mp4,60,63,1.05,1.53


In [8]:
#cell 3: kiểm tra API trích landmark của MediaPipe và tìm file mô hình hand_landmarker.task
import mediapipe as mp
from pathlib import Path

BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")

print("Có mp.solutions (cách cũ):", hasattr(mp, "solutions"))
try:
    from mediapipe.tasks.python import vision
    print("Có HandLandmarker (Tasks API):", hasattr(vision, "HandLandmarker"))
except Exception as e:
    print("Không import được Tasks API:", e)

print("File .task trong Data_321:", list(BASE.rglob("*.task")))

Có mp.solutions (cách cũ): False
Có HandLandmarker (Tasks API): True
File .task trong Data_321: [WindowsPath('C:/Users/Acer Nitro 5 Tiger/Downloads/Data_321/hand_landmarker.task')]


In [9]:
#cell 4: kiểm tra file mô hình và thử nạp HandLandmarker (chưa trích gì)
from pathlib import Path
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision

BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")
model_path = BASE / "hand_landmarker.task"
print("Dung lượng (MB):", round(model_path.stat().st_size / 1e6, 2))

opts = vision.HandLandmarkerOptions(
    base_options=mp_tasks.BaseOptions(model_asset_path=str(model_path)),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
)
detector = vision.HandLandmarker.create_from_options(opts)
print("Nạp mô hình OK")
detector.close()

Dung lượng (MB): 7.82
Nạp mô hình OK


In [10]:
#cell 5: trích landmark (21 điểm x,y,z) từ mọi video bằng MediaPipe (CPU), lưu mỗi video 1 file .npz vào Data_321\landmarks
import cv2, numpy as np, pandas as pd
import mediapipe as mp
from pathlib import Path
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision

BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")
OUT = BASE / "landmarks"
opts = vision.HandLandmarkerOptions(
    base_options=mp_tasks.BaseOptions(model_asset_path=str(BASE / "hand_landmarker.task")),
    running_mode=vision.RunningMode.VIDEO, num_hands=1)

def extract(path):
    det = vision.HandLandmarker.create_from_options(opts)   # tạo mới mỗi video để không giữ trạng thái cũ
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS)
    lm, hand, i = [], [], 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        r = det.detect_for_video(img, int(i * 1000 / fps))
        if r.hand_landmarks:
            lm.append([[p.x, p.y, p.z] for p in r.hand_landmarks[0]])
            hand.append(0 if r.handedness[0][0].category_name == "Left" else 1)
        else:
            lm.append(np.full((21, 3), np.nan)); hand.append(-1)   # mất tay: NaN, nhãn -1
        i += 1
    cap.release(); det.close()
    return np.array(lm, np.float32), np.array(hand, np.int8), fps

stats = []
for src in ["Dataset_hand", "dataset_labeled"]:
    for p in sorted((BASE / src).rglob("*.mp4")):
        lop = p.parent.name.lower()
        f = OUT / src / lop / (p.stem + ".npz")
        if not f.exists():
            f.parent.mkdir(parents=True, exist_ok=True)
            lm, hand, fps = extract(p)
            np.savez_compressed(f, lm=lm, hand=hand, fps=fps)
        else:
            lm = np.load(f)["lm"]
        stats.append((src, lop, np.isnan(lm[:, 0, 0]).mean()))

s = pd.DataFrame(stats, columns=["nguon", "lop", "ty_le_mat_tay"])
print(s.groupby(["nguon", "lop"]).agg(so_video=("lop", "count"), mat_tay_tb=("ty_le_mat_tay", "mean")).round(3))
print("Đã lưu tại:", OUT)

                          so_video  mat_tay_tb
nguon           lop                           
Dataset_hand    paper           27       0.000
                rock            27       0.044
                scissors        27       0.006
dataset_labeled paper           48       0.000
                rock            43       0.004
                scissors        43       0.000
Đã lưu tại: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\landmarks


In [11]:
#cell 6: nội suy landmark Dataset_hand từ 60 -> 120 fps bằng CUDA (chèn khung giữa), khung mất tay giữ NaN
import numpy as np, pandas as pd, torch
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA"
dev = "cuda"
BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")
SRC = BASE / "landmarks" / "Dataset_hand"
OUT = BASE / "landmarks_120" / "Dataset_hand"

def up2(lm, hand):
    a = torch.tensor(lm, device=dev); h = torch.tensor(hand, device=dev)
    new = torch.empty((2 * len(a) - 1, 21, 3), device=dev)
    new[0::2], new[1::2] = a, (a[:-1] + a[1:]) / 2        # khung giữa; NaN nếu 1 trong 2 khung mất tay
    hm = torch.where(h[:-1] == h[1:], h[:-1], torch.full_like(h[:-1], -1))
    nh = torch.empty(2 * len(h) - 1, dtype=h.dtype, device=dev)
    nh[0::2], nh[1::2] = h, hm
    return new.cpu().numpy(), nh.cpu().numpy()

rows = []
for f in sorted(SRC.rglob("*.npz")):
    d = np.load(f)
    lm, hand = up2(d["lm"], d["hand"])
    o = OUT / f.parent.name / f.name
    o.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(o, lm=lm, hand=hand, fps=120.0)
    rows.append((f.parent.name, f.name, len(d["lm"]), len(lm), np.isnan(d["lm"][:, 0, 0]).mean()))

r = pd.DataFrame(rows, columns=["lop", "file", "khung_60", "khung_120", "ty_le_mat_tay"])
print(r.groupby("lop").agg(so_video=("file", "count"), khung_60=("khung_60", "mean"),
                           khung_120=("khung_120", "mean")).round(1))
print("Video mất tay > 10%:"); print(r[r.ty_le_mat_tay > 0.1][["lop", "file", "ty_le_mat_tay"]])
print("Đã lưu tại:", OUT.parent)

          so_video  khung_60  khung_120
lop                                    
paper           27      63.5      126.0
rock            27      63.6      126.3
scissors        27      64.2      127.4
Video mất tay > 10%:
         lop                       file  ty_le_mat_tay
38      rock  VID_20261002111612136.npz       0.278689
39      rock  VID_20261002111704795.npz       0.287879
41      rock  VID_20261002111933974.npz       0.225806
42      rock  VID_20261002112021578.npz       0.322581
66  scissors  VID_20261002112614253.npz       0.161290
Đã lưu tại: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\landmarks_120


In [12]:
#cell 7: (2c) gộp 2 nguồn thành bảng manifest.csv: đường dẫn landmark 120 fps, nguồn, lớp (chữ thường), nhãn số, số khung
import numpy as np, pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")
roots = {"Dataset_hand": BASE / "landmarks_120" / "Dataset_hand",
         "dataset_labeled": BASE / "landmarks" / "dataset_labeled"}
LABEL = {"paper": 0, "rock": 1, "scissors": 2}

rows = []
for nguon, root in roots.items():
    for f in sorted(root.rglob("*.npz")):
        d = np.load(f)
        rows.append(dict(path=str(f.relative_to(BASE)), nguon=nguon, lop=f.parent.name,
                         nhan=LABEL[f.parent.name], khung=len(d["lm"]), fps=float(d["fps"])))

m = pd.DataFrame(rows)
m.to_csv(BASE / "manifest.csv", index=False)
print(m.groupby(["nguon", "lop"]).agg(so_video=("path", "count"), khung_tb=("khung", "mean"),
                                      fps=("fps", "median")).round(1))
print("Tổng:", len(m), "video | lưu tại:", BASE / "manifest.csv")

                          so_video  khung_tb    fps
nguon           lop                                
Dataset_hand    paper           27     126.0  120.0
                rock            27     126.3  120.0
                scissors        27     127.4  120.0
dataset_labeled paper           48     112.0  120.0
                rock            43     122.2  120.0
                scissors        43     142.5  120.0
Tổng: 215 video | lưu tại: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\manifest.csv


In [13]:
#cell 8: (2d) chia train/val/test = 70/15/15 theo video, giữ tỉ lệ nguồn x lớp, ghi cột "tap" vào manifest.csv
import numpy as np, pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")
m = pd.read_csv(BASE / "manifest.csv")
rng = np.random.default_rng(42)          # cố định seed để chạy lại ra đúng kết quả cũ

m["tap"] = ""
for _, g in m.groupby(["nguon", "lop"]):
    idx = rng.permutation(g.index.to_numpy())
    n_tr, n_va = round(len(idx) * 0.70), round(len(idx) * 0.15)
    m.loc[idx[:n_tr], "tap"] = "train"
    m.loc[idx[n_tr:n_tr + n_va], "tap"] = "val"
    m.loc[idx[n_tr + n_va:], "tap"] = "test"

m.to_csv(BASE / "manifest.csv", index=False)
print(m.pivot_table(index=["nguon", "lop"], columns="tap", values="path", aggfunc="count")[["train", "val", "test"]])
print(m["tap"].value_counts())

tap                       train  val  test
nguon           lop                       
Dataset_hand    paper        19    4     4
                rock         19    4     4
                scissors     19    4     4
dataset_labeled paper        34    7     7
                rock         30    6     7
                scissors     30    6     7
tap
train    151
test      33
val       31
Name: count, dtype: int64
